In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

df = pd.read_csv("squat_dataset_3class.csv")

df["rep_id"] = (df["frame_time"] == 0).cumsum()

df["depth_flag"] = (df["knee_angle"] < 100).astype(int)
df["forward_lean_flag"] = (df["torso_angle"] > 35).astype(int)

df["movement_speed"] = df.groupby("rep_id")["knee_angle"].diff().abs().fillna(0)

df["knee_stability"] = (
    df.groupby("rep_id")["knee_angle"]
    .rolling(3)
    .std()
    .reset_index(level=0, drop=True)
    .fillna(0)
)

df["hip_knee_diff"] = abs(df["hip_angle"] - df["knee_angle"])

# 🔥 NEW FEATURES
df["acceleration"] = df.groupby("rep_id")["movement_speed"].diff().abs().fillna(0)
df["stability_ratio"] = df["knee_stability"] / (df["knee_angle"] + 1e-6)

X = df[[
    "knee_angle",
    "hip_angle",
    "ankle_angle",
    "torso_angle",
    "depth_flag",
    "forward_lean_flag",
    "movement_speed",
    "knee_stability",
    "hip_knee_diff",
    "acceleration",
    "stability_ratio"
]]

y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 0.8

Classification Report:
                  precision    recall  f1-score   support

        correct       0.86      0.75      0.80         8
incorrect_major       0.70      1.00      0.82         7
incorrect_minor       1.00      0.60      0.75         5

       accuracy                           0.80        20
      macro avg       0.85      0.78      0.79        20
   weighted avg       0.84      0.80      0.80        20


Confusion Matrix:
 [[6 2 0]
 [0 7 0]
 [1 1 3]]


In [36]:
def give_feedback(pred):
    if pred == "correct":
        return "Good squat 👍"
    elif pred == "incorrect_major":
        return "Go deeper or keep your chest up ❗"
    else:
        return "Control movement / stabilize knees ⚠️"

In [40]:
import numpy as np

sample = np.array([[
    90,   # knee_angle
    95,   # hip_angle
    65,   # ankle_angle
    20,   # torso_angle
    1,    # depth_flag (knee < 100)
    0,    # forward_lean_flag (torso <= 35)
    0,    # movement_speed
    0,    # knee_stability
    5,    # hip_knee_diff
    0,    # acceleration
    0     # stability_ratio
]])

prediction = model.predict(sample)
give_feedback(pred)

print("Prediction:", prediction[0])

SyntaxError: invalid syntax (773491232.py, line 17)

In [38]:
import numpy as np

sample = np.array([[
120, 120, 75, 25, 0, 0, 0, 0, 0, 0, 0]])

prediction = model.predict(sample)

print("Prediction:", prediction[0])

Prediction: incorrect_major


C:\Users\jhanv\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [39]:
import numpy as np

sample = np.array([[95, 95, 65, 50, 1, 1, 0, 0, 0, 0, 0]])

prediction = model.predict(sample)

print("Prediction:", prediction[0])

Prediction: incorrect_major


C:\Users\jhanv\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
